In [1]:
from pathlib import Path
from collections import Counter
import re

import numpy as np
import pandas as pd
from IPython.display import display

# Find the project root automatically
current_path = Path.cwd()

PROJECT_ROOT = next(
    (p for p in [current_path, *current_path.parents]
     if (p / "requirements.txt").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root. Make sure this notebook is inside "
        "the ThreatTrace project."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "cicids2017"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_FILE = PROCESSED_DIR / "cicids2017_cleaned.csv"

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {RAW_DIR}")

csv_files = sorted(RAW_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {RAW_DIR}")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("Number of CSV files:", len(csv_files))
print("Output file:", OUTPUT_FILE)

for file in csv_files:
    print(f"{file.name}: {file.stat().st_size / (1024**2):.2f} MB")

Project root: d:\ThreatTrace
Raw data folder: d:\ThreatTrace\data\raw\cicids2017
Number of CSV files: 8
Output file: d:\ThreatTrace\data\processed\cicids2017_cleaned.csv
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 73.55 MB
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 73.34 MB
Friday-WorkingHours-Morning.pcap_ISCX.csv: 55.62 MB
Monday-WorkingHours.pcap_ISCX.csv: 168.73 MB
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 79.25 MB
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 49.61 MB
Tuesday-WorkingHours.pcap_ISCX.csv: 128.82 MB
Wednesday-workingHours.pcap_ISCX.csv: 214.74 MB


In [2]:
schemas = {}

for file in csv_files:
    columns = pd.read_csv(
        file,
        nrows=0,
        encoding="utf-8",
        encoding_errors="replace"
    ).columns.tolist()

    # Remove leading and trailing spaces from column names
    normalized_columns = [str(col).strip() for col in columns]

    # Detect names that become duplicates after trimming
    if len(normalized_columns) != len(set(normalized_columns)):
        raise ValueError(
            f"Duplicate column names after trimming in {file.name}"
        )

    schemas[file.name] = normalized_columns

reference_file = csv_files[0].name
reference_schema = schemas[reference_file]

for filename, columns in schemas.items():
    matches = columns == reference_schema
    print(f"{filename}: {len(columns)} columns | Matches reference: {matches}")

    if not matches:
        missing = sorted(set(reference_schema) - set(columns))
        extra = sorted(set(columns) - set(reference_schema))

        print("  Missing columns:", missing)
        print("  Extra columns:", extra)

        if not missing and not extra:
            print("  Column order differs.")

if not all(columns == reference_schema for columns in schemas.values()):
    raise ValueError(
        "Schema mismatch detected. Resolve it before combining the files."
    )

# Identify the target label column
label_matches = [
    col for col in reference_schema
    if col.strip().casefold() == "label"
]

if len(label_matches) != 1:
    raise ValueError(
        f"Expected one Label column, found: {label_matches}"
    )

LABEL_COL = label_matches[0]

print("\nSchema validation passed.")
print("Number of features including label:", len(reference_schema))
print("Target column:", LABEL_COL)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 79 columns | Matches reference: True
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 79 columns | Matches reference: True
Friday-WorkingHours-Morning.pcap_ISCX.csv: 79 columns | Matches reference: True
Monday-WorkingHours.pcap_ISCX.csv: 79 columns | Matches reference: True
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 79 columns | Matches reference: True
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 79 columns | Matches reference: True
Tuesday-WorkingHours.pcap_ISCX.csv: 79 columns | Matches reference: True
Wednesday-workingHours.pcap_ISCX.csv: 79 columns | Matches reference: True

Schema validation passed.
Number of features including label: 79
Target column: Label


In [3]:
def clean_column_name(column):
    """Standardize column names without changing their meaning."""
    column = str(column).strip()
    column = re.sub(r"\s+", " ", column)
    return column


def clean_label(label):
    """Trim and standardize known CICIDS2017 attack labels."""
    if pd.isna(label):
        return np.nan

    label = str(label).strip()
    label = re.sub(r"\s+", " ", label)

    # Standardize common encoding-corrupted Web Attack labels
    normalized = label.casefold()
    normalized = normalized.replace("\ufffd", "-")
    normalized = re.sub(r"\s*-\s*", " - ", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()

    known_labels = {
        "web attack - brute force": "Web Attack - Brute Force",
        "web attack - xss": "Web Attack - XSS",
        "web attack - sql injection": "Web Attack - SQL Injection",
    }

    return known_labels.get(normalized, label)


def clean_chunk(df):
    """Apply consistent cleaning to one chunk of records."""
    df.columns = [clean_column_name(col) for col in df.columns]

    if df.columns.duplicated().any():
        raise ValueError("Duplicate columns detected after normalization.")

    # Locate the target column after normalizing column names
    label_candidates = [
        col for col in df.columns
        if col.casefold() == "label"
    ]

    if len(label_candidates) != 1:
        raise ValueError(
            f"Expected one Label column, found {label_candidates}"
        )

    label_col = label_candidates[0]

    df[label_col] = df[label_col].map(clean_label)

    # Replace positive and negative infinity in numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if len(numeric_cols):
        df[numeric_cols] = df[numeric_cols].replace(
            [np.inf, -np.inf], np.nan
        )

    return df

In [4]:
CHUNK_SIZE = 100_000

# Remove an old output to prevent accidentally appending to previous results
if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

total_rows = 0
total_missing_labels = 0
total_infinite_values = 0

label_counts = Counter()
file_summaries = []

for file_index, file in enumerate(csv_files, start=1):
    print(f"\n[{file_index}/{len(csv_files)}] Processing: {file.name}")

    file_rows = 0
    file_missing_labels = 0
    file_labels = Counter()

    reader = pd.read_csv(
        file,
        chunksize=CHUNK_SIZE,
        low_memory=False,
        encoding="utf-8",
        encoding_errors="replace"
    )

    for chunk in reader:
        # Count infinite values before replacing them
        numeric_cols = chunk.select_dtypes(include=[np.number]).columns
        infinite_count = int(
            np.isinf(chunk[numeric_cols].to_numpy()).sum()
        ) if len(numeric_cols) else 0

        total_infinite_values += infinite_count

        chunk = clean_chunk(chunk)

        label_col = next(
            col for col in chunk.columns
            if col.casefold() == "label"
        )

        file_rows += len(chunk)
        total_rows += len(chunk)

        missing_labels = int(chunk[label_col].isna().sum())
        file_missing_labels += missing_labels
        total_missing_labels += missing_labels

        counts = chunk[label_col].value_counts(dropna=False)

        for label, count in counts.items():
            display_label = "<MISSING>" if pd.isna(label) else str(label)
            file_labels[display_label] += int(count)
            label_counts[display_label] += int(count)

        # Write the first chunk with a header; append subsequent chunks
        # and files without repeating the header.
        chunk.to_csv(
            OUTPUT_FILE,
            mode="w" if total_rows == len(chunk) else "a",
            header=(total_rows == len(chunk)),
            index=False
        )

    file_summaries.append({
        "file": file.name,
        "rows": file_rows,
        "missing_labels": file_missing_labels,
        "labels_found": len(file_labels)
    })

    print(f"Rows processed: {file_rows:,}")
    print(f"Missing labels: {file_missing_labels:,}")

print("\nPreprocessing finished.")
print(f"Total rows processed: {total_rows:,}")
print(f"Total missing labels: {total_missing_labels:,}")
print(f"Total infinite values replaced: {total_infinite_values:,}")
print(f"Output size: {OUTPUT_FILE.stat().st_size / (1024**2):.2f} MB")


[1/8] Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Rows processed: 225,745
Missing labels: 0

[2/8] Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Rows processed: 286,467
Missing labels: 0

[3/8] Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Rows processed: 191,033
Missing labels: 0

[4/8] Processing: Monday-WorkingHours.pcap_ISCX.csv
Rows processed: 529,918
Missing labels: 0

[5/8] Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Rows processed: 288,602
Missing labels: 0

[6/8] Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Rows processed: 170,366
Missing labels: 0

[7/8] Processing: Tuesday-WorkingHours.pcap_ISCX.csv
Rows processed: 445,909
Missing labels: 0

[8/8] Processing: Wednesday-workingHours.pcap_ISCX.csv
Rows processed: 692,703
Missing labels: 0

Preprocessing finished.
Total rows processed: 2,830,743
Total missing labels: 0
Total infinite values replaced: 4,376
Output size: 921.43 MB


In [5]:
summary_df = pd.DataFrame(file_summaries)

display(
    summary_df.style.format({
        "rows": "{:,}",
        "missing_labels": "{:,}"
    })
)

print("Total rows:", f"{summary_df['rows'].sum():,}")
print("Files processed:", len(summary_df))

,file,rows,missing_labels,labels_found
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,"225,745",0,2
1,Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv,"286,467",0,2
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,"191,033",0,2
3,Monday-WorkingHours.pcap_ISCX.csv,"529,918",0,1
4,Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv,"288,602",0,2
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv,"170,366",0,4
6,Tuesday-WorkingHours.pcap_ISCX.csv,"445,909",0,3
7,Wednesday-workingHours.pcap_ISCX.csv,"692,703",0,6


Total rows: 2,830,743
Files processed: 8


In [6]:
label_distribution = pd.DataFrame(
    [
        {"Label": label, "Count": count}
        for label, count in label_counts.items()
    ]
).sort_values("Count", ascending=False)

label_distribution["Percentage"] = (
    label_distribution["Count"] / total_rows * 100
).round(4)

display(label_distribution)

print("Unique labels:", len(label_distribution))

,Label,Count,Percentage
1,BENIGN,2273097,80.3004
10,DoS Hulk,231073,8.1630
2,PortScan,158930,5.6144
0,DDoS,128027,4.5227
13,DoS GoldenEye,10293,0.3636
8,FTP-Patator,7938,0.2804
9,SSH-Patator,5897,0.2083
11,DoS slowloris,5796,0.2048
12,DoS Slowhttptest,5499,0.1943
3,Bot,1966,0.0695


Unique labels: 15


In [7]:
# Read only a sample to verify the output schema
processed_columns = pd.read_csv(
    OUTPUT_FILE,
    nrows=0,
    encoding="utf-8"
).columns.tolist()

assert len(processed_columns) == len(reference_schema)
assert processed_columns == reference_schema
assert OUTPUT_FILE.exists() and OUTPUT_FILE.stat().st_size > 0

# Count rows in the processed file without loading it all into memory
validated_rows = 0
validated_labels = Counter()

for chunk in pd.read_csv(
    OUTPUT_FILE,
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    validated_rows += len(chunk)

    label_col = next(
        col for col in chunk.columns
        if col.casefold() == "label"
    )

    for label, count in chunk[label_col].value_counts(dropna=False).items():
        key = "<MISSING>" if pd.isna(label) else str(label)
        validated_labels[key] += int(count)

assert validated_rows == total_rows, (
    f"Row mismatch: processed {total_rows:,}, "
    f"validated {validated_rows:,}"
)

assert validated_labels == label_counts, (
    "Label distribution mismatch between processing and validation."
)

print("Validation passed.")
print(f"Rows in output: {validated_rows:,}")
print(f"Columns in output: {len(processed_columns)}")
print(f"Output file: {OUTPUT_FILE}")
print("Output exists and is non-empty.")

Validation passed.
Rows in output: 2,830,743
Columns in output: 79
Output file: d:\ThreatTrace\data\processed\cicids2017_cleaned.csv
Output exists and is non-empty.


In [8]:
missing_counts = Counter()
checked_rows = 0

for chunk in pd.read_csv(
    OUTPUT_FILE,
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    checked_rows += len(chunk)
    missing = chunk.isna().sum()

    for column, count in missing.items():
        missing_counts[column] += int(count)

missing_df = pd.DataFrame({
    "Column": list(missing_counts.keys()),
    "Missing_Count": list(missing_counts.values())
})

missing_df["Missing_Percentage"] = (
    missing_df["Missing_Count"] / checked_rows * 100
).round(4)

display(
    missing_df.sort_values("Missing_Count", ascending=False)
              .head(20)
)

print(f"Rows checked: {checked_rows:,}")
print("No missing values were automatically dropped.")

,Column,Missing_Count,Missing_Percentage
14,Flow Bytes/s,2867,0.1013
15,Flow Packets/s,2867,0.1013
1,Flow Duration,0,0.0000
3,Total Backward Packets,0,0.0000
4,Total Length of Fwd Packets,0,0.0000
5,Total Length of Bwd Packets,0,0.0000
0,Destination Port,0,0.0000
6,Fwd Packet Length Max,0,0.0000
7,Fwd Packet Length Min,0,0.0000
9,Fwd Packet Length Std,0,0.0000


Rows checked: 2,830,743
No missing values were automatically dropped.


In [9]:
summary_path = PROCESSED_DIR / "preprocessing_summary.txt"

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("CICIDS2017 Preprocessing Summary\n")
    f.write("=" * 40 + "\n")
    f.write(f"Input files: {len(csv_files)}\n")
    f.write(f"Rows processed: {total_rows:,}\n")
    f.write(f"Columns: {len(processed_columns)}\n")
    f.write(f"Missing labels: {total_missing_labels:,}\n")
    f.write(f"Infinite values replaced: {total_infinite_values:,}\n")
    f.write(f"Output file: {OUTPUT_FILE.name}\n\n")
    f.write("Attack-label distribution:\n")

    for label, count in label_counts.most_common():
        f.write(f"{label}: {count:,}\n")

print("Summary saved to:", summary_path)

Summary saved to: d:\ThreatTrace\data\processed\preprocessing_summary.txt
